# Tests et sélection des prompts

## Objectif du notebook

Ce notebook a pour objectif de concevoir, tester et comparer plusieurs prompts destinés à interroger des grands modèles de langage sur les questions médicales MCQU préparées dans le notebook précédent.

Les expériences sont réalisées sur un échantillon reproductible du split `validation`. Le split `test` n’est pas utilisé pendant cette phase afin de le réserver à l’évaluation finale.

Le notebook réalise les étapes suivantes :

1. charger le dataset MCQU préparé ;
2. sélectionner un échantillon de validation reproductible ;
3. définir plusieurs versions de prompts ;
4. construire les messages envoyés aux LLM ;
5. interroger plusieurs modèles dans des conditions identiques ;
6. extraire la lettre et la justification générées ;
7. mesurer la validité du format et l’exactitude des réponses ;
8. sélectionner le prompt retenu pour le benchmark final.

Le prompt sélectionné devra produire une réponse structurée comportant :

- une lettre parmi `A`, `B`, `C`, `D` ou `E` ;
- une justification médicale concise ;
- aucune information extérieure à la question lorsque celle-ci n’est pas nécessaire.

## Pipeline expérimental

```mermaid
flowchart TD
    A["MCQU validation préparé"] --> B["Échantillon reproductible"]
    B --> C["Versions des prompts"]
    C --> D["Appels aux LLM"]
    D --> E["Réponses brutes"]
    E --> F["Extraction lettre et justification"]
    F --> G["Évaluation des prompts"]
    G --> H["Sélection du prompt final"]

# 1. Importations et chemins

In [23]:
from pathlib import Path
import json
import re
import time

import pandas as pd

In [24]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
PROMPT_RESULTS_DIR = RESULTS_DIR / "prompt_tests"

PROMPT_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

## 2. Chargement de la validation

In [25]:
VALIDATION_PATH = (
    PROCESSED_DIR
    / "benchmark_mcqu_validation.parquet"
)

df_validation = pd.read_parquet(
    VALIDATION_PATH
)

print("Dimensions :", df_validation.shape)

# affichage des 5 premières lignes du DataFrame
display(
    df_validation[
        [
            "sample_id",
            "medical_subject",
            "question_type",
            "reference_letter",
            "question_context",
        ]
    ].head()
)

Dimensions : (2561, 19)


,sample_id,medical_subject,question_type,reference_letter,question_context
0,mcqu_validation_9940,Ophthalmology,Understanding,C,Question :\nL'amblyopie fonctionnelle se défin...
1,mcqu_validation_14132,Epidemiology,Reasoning,D,"Question :\nDe 1967 à 1972, un groupe de 300 o..."
2,mcqu_validation_23706,Psychiatry,Understanding,E,Question :\n(cochez la réponse fausse) Un synd...
3,mcqu_validation_25123,Physiology,Understanding,E,Question :\n(cochez la réponse fausse) Le diag...
4,mcqu_validation_3365,Rheumatology,Understanding,D,Question :\nAu cours de la détection d'anticor...


In [80]:
print(df_validation.loc[6, "question_context"])

Question :
L'état mixte est un état pathologique marqué par une des 5 propositions suivantes :

Propositions :
A. Intensité des hallucinations
B. Alternance de dépression et d'excitation
C. Coexistence de thèmes dépressifs et de thèmes maniaques
D. Coexistence de traits névrotiques et de traits psychotiques
E. Association d'une dépression et d'une détérioration


## 3. Echantillonage

Pour les premiers tests, 100 questions suffisent. Elles permettent de comparer les prompts sans multiplier inutilement les appels.

In [26]:
RANDOM_SEED = 42
PROMPT_SAMPLE_SIZE = 100

In [27]:
df_prompt_sample = (
    df_validation.sample(
        n=PROMPT_SAMPLE_SIZE,
        random_state=RANDOM_SEED
    )
)

In [69]:
display(df_prompt_sample.head())

,sample_id,id,configuration,split,clinical_case,question,answer_a,answer_b,answer_c,answer_d,answer_e,choices,choices_text,reference_letter,reference_answer,medical_subject,question_type,task,question_context
2157,mcqu_validation_18257,18257,mcqu,validation,"Une patiente de 26 ans, enceinte de 20 semaine...",Les risques de toxoplasmose congénitale pour l...,90%,70%,50%,20%,"0,5%","{'A': '90%', 'B': '70%', 'C': '50%', 'D': '20%...","A. 90%\nB. 70%\nC. 50%\nD. 20%\nE. 0,5%",D,20%,Gynecology and Obstetrics,Reasoning,QCU,"Cas clinique :\nUne patiente de 26 ans, encein..."
1738,mcqu_validation_7122,7122,mcqu,validation,Vous êtes appelé en urgence en novembre à 8 he...,Ce diagnostic est spécialement fondé sur un ou...,Le caractère collectif de l'intoxication,Le signe de Babinski droit,L'absence de convulsions chez les trois victimes,L'anomalie pupillaire chez la femme,L'absence de cyanose chez les trois victimes,{'A': 'Le caractère collectif de l'intoxicatio...,A. Le caractère collectif de l'intoxication\nB...,A,Le caractère collectif de l'intoxication,Occupational Medicine,Reasoning,QCU,Cas clinique :\nVous êtes appelé en urgence en...
1173,mcqu_validation_5451,5451,mcqu,validation,Une femme de 25 ans consulte pour des lésions ...,"Parmi les diagnostics suivants, quel est le pl...",Erythème noueux,Panniculite,Phlébites superficielles,Psoriasis,Erythème polymorphe,"{'A': 'Erythème noueux', 'B': 'Panniculite', '...",A. Erythème noueux\nB. Panniculite\nC. Phlébit...,A,Erythème noueux,Dermatology,Understanding,QCU,Cas clinique :\nUne femme de 25 ans consulte p...
478,mcqu_validation_25802,25802,mcqu,validation,"Madame F.Z. 24 ans, à déjà accouché d'un préma...",(cochez la réponse fausse) Quels sont les exam...,Enregistrement du rythme cardiaque foetal,E.C.B.U.,Bactériologie des pertes vaginales et cervicales,Morphogramme à l'échographie,Localisation placentaire à l'échographie,{'A': 'Enregistrement du rythme cardiaque foet...,A. Enregistrement du rythme cardiaque foetal\n...,D,Morphogramme à l'échographie,Gynecology and Obstetrics,Understanding,QCU,"Cas clinique :\nMadame F.Z. 24 ans, à déjà acc..."
1356,mcqu_validation_24634,24634,mcqu,validation,,(cochez la réponse fausse) Devant une fièvre a...,Diarrhée,Dissociation pouls et température,Tuphos,Angine de Vincent,Taches rosées lenticulaires,"{'A': 'Diarrhée', 'B': 'Dissociation pouls et ...",A. Diarrhée\nB. Dissociation pouls et températ...,D,Angine de Vincent,Infectious Diseases,Understanding,QCU,Question :\n(cochez la réponse fausse) Devant ...


Nous allons sauvegarder cet échantillon afin de l'utiliser à l'indentique

In [28]:
SAMPLE_PATH = (
    PROCESSED_DIR
    / "prompt_validation_sample.parquet"
)

df_prompt_sample.to_parquet(
    SAMPLE_PATH,
    index=False
)

print("Échantillon enregistré :", SAMPLE_PATH)

Échantillon enregistré : c:\Users\MANEL\Dropbox\projet_evaluation_LLM_medicale\data\processed\prompt_validation_sample.parquet


## 4. Création de prompts
Dans cette section, nous allons créer plusieurs prompts des questions à choix multiples. Les prompts seront utilisés pour interroger un modèle de LLM afin d'obtenir une réponse à la question posée. Les prompt seront construits en combinant le contexte de la question, les propositions et une instruction pour le modèle. Nous avons établie 3 prompts différents pour tester les performances du modèle sur la tâche de question à choix unique. Chaque prompt a un niveau de détail croissant, allant d'une simple instruction à une demande d'analyse plus approfondie.


In [29]:
# Prompt à réponse direct (sans justification ni confiance)
PROMPT_V1 = """
Vous devez répondre à une question médicale à choix unique.

{question_context}

Sélectionnez une seule proposition parmi A, B, C, D ou E.

Répondez uniquement avec la lettre correspondante.
""".strip()

In [30]:
# Prompt à réponse avec justification (sans confiance)
PROMPT_V2 = """
Vous devez répondre à une question médicale à choix unique.

{question_context}

Sélectionnez une seule proposition parmi A, B, C, D ou E.

Répondez exactement au format suivant :

Réponse : <lettre>
Justification : <explication médicale concise>
""".strip()

In [31]:
# Prompt à réponse avec justification et confiance
PROMPT_V3 = """
Vous devez répondre à une question médicale à choix unique.

{question_context}

Analysez uniquement les informations utiles à la résolution de la question. N’inventez aucune donnée clinique absente du cas présenté.

Sélectionnez une seule proposition parmi A, B, C, D ou E.

Répondez exactement au format suivant :

Réponse : <lettre>
Justification : <explication médicale concise>
Confiance : <nombre entier compris entre 0 et 100>
""".strip()

In [32]:
# regroupement des prompts dans un dictionnaire
PROMPT_TEMPLATES = {
    "prompt_v1": PROMPT_V1,
    "prompt_v2": PROMPT_V2,
    "prompt_v3": PROMPT_V3,
}

## 5. Construction des messages

In [33]:
# Fonction pour construire le prompt final en insérant le contexte de la question
def build_prompt(
    question_context,
    prompt_template,
):
    return prompt_template.format(
        question_context=question_context
    )

In [ ]:
# test sur un exemple
example_row = df_prompt_sample.iloc[0]

example_prompt = build_prompt(
    question_context=example_row[
        "question_context"
    ],
    prompt_template=PROMPT_V2,
)

print(example_prompt)

Vous devez répondre à une question médicale à choix unique.

Cas clinique :
Une patiente de 26 ans, enceinte de 20 semaines, vient en consultation pour sa visite du 5 ème mois. Le sérodiagnostic de toxoplasmose pratiqué 15 jours auparavant objective un taux d'lgG à 450 U.I., présence d'lgM. Le précédent sérodiagnostic pratiqué en début de grossesse à 6 semaines d'aménorrhée était négatif. Il s'agit d'une séroconversion. Un traitement à la Spiramycine (3 g/j) est instauré

Question :
Les risques de toxoplasmose congénitale pour le foetus sont de l'ordre de :

Propositions :
A. 90%
B. 70%
C. 50%
D. 20%
E. 0,5%

Sélectionnez une seule proposition parmi A, B, C, D ou E.

Répondez exactement au format suivant :

Réponse : <lettre>
Justification : <explication médicale concise>


## 6. Tableau expériemental
Nous allons créer un tableau expérimental ayant pour chaque ligne de l'échantillon, les 3 prompts différents à tester.



In [35]:
experiment_rows = []

for _, row in df_prompt_sample.iterrows():
    for prompt_version, template in (
        PROMPT_TEMPLATES.items()
    ):
        experiment_rows.append(
            {
                "sample_id": row["sample_id"],
                "prompt_version": prompt_version,
                "prompt_text": build_prompt(
                    row["question_context"],
                    template,
                ),
                "reference_letter": row[
                    "reference_letter"
                ],
                "reference_answer": row[
                    "reference_answer"
                ],
                "medical_subject": row[
                    "medical_subject"
                ],
                "question_type": row[
                    "question_type"
                ],
            }
        )

df_experiments = pd.DataFrame(
    experiment_rows
)

In [68]:
display(df_experiments.head())

,sample_id,prompt_version,prompt_text,reference_letter,reference_answer,medical_subject,question_type
0,mcqu_validation_18257,prompt_v1,Vous devez répondre à une question médicale à ...,D,20%,Gynecology and Obstetrics,Reasoning
1,mcqu_validation_18257,prompt_v2,Vous devez répondre à une question médicale à ...,D,20%,Gynecology and Obstetrics,Reasoning
2,mcqu_validation_18257,prompt_v3,Vous devez répondre à une question médicale à ...,D,20%,Gynecology and Obstetrics,Reasoning
3,mcqu_validation_7122,prompt_v1,Vous devez répondre à une question médicale à ...,A,Le caractère collectif de l'intoxication,Occupational Medicine,Reasoning
4,mcqu_validation_7122,prompt_v2,Vous devez répondre à une question médicale à ...,A,Le caractère collectif de l'intoxication,Occupational Medicine,Reasoning


## 7. Colonnes à enregistrer

Nous allons retourner certaines informations afin de distinguer : 
- la réponse brute du modèle 
- les informations extraites 
- la réponse attendue 
- le résultat de l’évaluation 
- les erreurs techniques. 


In [36]:

result_columns = [
    "sample_id",
    "model_name",
    "model_version",
    "prompt_version",
    "prompt_text",
    "temperature",
    "raw_response",
    "predicted_letter",
    "generated_justification",
    "declared_confidence",
    "response_format_valid",
    "reference_letter",
    "is_correct",
    "latency_seconds",
    "generation_error",
]

# 8. Extraction des réponses, des justification et des confiances
Avant les appels aux modèles, préparez la fonction de lecture des réponses :

In [37]:
VALID_LETTERS = {"A", "B", "C", "D", "E"}

In [38]:
def extract_predicted_letter(response):
    if not isinstance(response, str):
        return None

    patterns = [
        r"Réponse\s*:\s*([A-E])",
        r"^\s*([A-E])\s*$",
        r"^\s*([A-E])[\.\)]",
    ]

    for pattern in patterns:
        match = re.search(
            pattern,
            response,
            flags=re.IGNORECASE,
        )

        if match:
            return match.group(1).upper()

    return None

In [39]:
test_responses = [
    "Réponse : C",
    "Réponse : B\nJustification : ...",
    "A",
    "D. La réponse est...",
    "Je ne sais pas",
    "Confiance = 80"
]

for response in test_responses:
    print(
        response,
        "→",
        extract_predicted_letter(response)
    )

Réponse : C → C
Réponse : B
Justification : ... → B
A → A
D. La réponse est... → D
Je ne sais pas → None
Confiance = 80 → None


In [40]:
# extraction de la justification
def extract_justification(response):
    if not isinstance(response, str):
        return None

    match = re.search(
        r"Justification\s*:\s*(.*?)(?:\nConfiance\s*:|$)",
        response,
        flags=re.IGNORECASE | re.DOTALL,
    )

    if not match:
        return None

    justification = match.group(1).strip()

    return justification or None

In [41]:
# extraction de la confiance
def extract_confidence(response):
    if not isinstance(response, str):
        return None

    match = re.search(
        r"Confiance\s*[:=]\s*(\d{1,3})",
        response,
        flags=re.IGNORECASE,
    )

    if not match:
        return None

    confidence = int(match.group(1))

    if 0 <= confidence <= 100:
        return confidence

    return None

## 9. Analyse et validation des réponses

Cette fonction regroupe les informations extraites de la réponse brute et vérifie si le modèle a respecté le format demandé par le prompt.

In [42]:
def parse_model_response(
    response,
    prompt_version,
):
    predicted_letter = extract_predicted_letter(
        response
    )

    generated_justification = extract_justification(
        response
    )

    declared_confidence = extract_confidence(
        response
    )

    # Chaque prompt demande un format différent
    if prompt_version == "prompt_v1":
        response_format_valid = (
            predicted_letter is not None
        )

    elif prompt_version == "prompt_v2":
        response_format_valid = (
            predicted_letter is not None
            and generated_justification is not None
        )

    elif prompt_version == "prompt_v3":
        response_format_valid = (
            predicted_letter is not None
            and generated_justification is not None
            and declared_confidence is not None
        )

    else:
        response_format_valid = False

    return {
        "predicted_letter": predicted_letter,
        "generated_justification": (
            generated_justification
        ),
        "declared_confidence": declared_confidence,
        "response_format_valid": (
            response_format_valid
        ),
    }

## 10 Importation d'un LLM et test
Nous allons importer l'API Gemini de Google pour interroger le modèle de language


In [47]:
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from google.genai import types

In [53]:
print(PROJECT_ROOT)

c:\Users\MANEL\Dropbox\projet_evaluation_LLM_medicale


In [56]:
# Chargement de la clé API du fichier .env


env_path = Path.cwd()/ ".env"

load_dotenv(env_path)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if GEMINI_API_KEY is None:
    raise ValueError("GEMINI_API_KEY introuvable")

print("Clé Gemini chargée avec succès.")

Clé Gemini chargée avec succès.


In [57]:
# initialisation client Gemini

gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

## 11. Fonction d’appel à Gemini
La fonction suivante envoie un prompt au modèle Gemini et retourne la réponse brute, le temps d’exécution et l’éventuelle erreur technique.

In [61]:
def call_gemini(
    prompt,
    model_name="gemini-3.6-flash",
):
    start_time = time.perf_counter()

    try:
        interaction = (
            gemini_client.interactions.create(
                model=model_name,
                input=prompt,
                store=False,
            )
        )

        raw_response = interaction.output_text
        generation_error = None

    except Exception as error:
        raw_response = None

        generation_error = (
            f"{type(error).__name__}: {error}"
        )

    latency_seconds = (
        time.perf_counter() - start_time
    )

    return {
        "raw_response": raw_response,
        "latency_seconds": latency_seconds,
        "generation_error": generation_error,
    }

In [64]:

# test du prompt sur un exemple
test_result = call_gemini(
    prompt=example_prompt,
)

print(
    "Réponse brute :\n",
    test_result["raw_response"]
)

print(
    "\nLatence :",
    round(
        test_result["latency_seconds"],
        2,
    ),
    "secondes"
)

print(
    "\nErreur :",
    test_result["generation_error"]
)

Réponse brute :
 Réponse : D
Justification : Le risque de transmission materno-fœtale de la toxoplasmose augmente avec l'âge gestationnel : il est faible au 1er trimestre (~10-15 %), de l'ordre de 20 à 30 % au milieu de la grossesse (vers 20 semaines d'aménorrhée), et atteint 60 à 80 % au 3ème trimestre. Ainsi, à 20 SA, le risque d'atteinte fœtale est d'environ 20 %.

Latence : 60.63 secondes

Erreur : None


In [ ]:
# comparaison de la réponse brute avec le format attendu avec la fonction parse_model_response
parsed_test_result = parse_model_response(
    response=test_result["raw_response"],
    prompt_version="prompt_v2",
)

parsed_test_result

{'predicted_letter': 'D',
 'generated_justification': "Le risque de transmission materno-fœtale de la toxoplasmose augmente avec l'âge gestationnel : il est faible au 1er trimestre (~10-15 %), de l'ordre de 20 à 30 % au milieu de la grossesse (vers 20 semaines d'aménorrhée), et atteint 60 à 80 % au 3ème trimestre. Ainsi, à 20 SA, le risque d'atteinte fœtale est d'environ 20 %.",
 'declared_confidence': None,
 'response_format_valid': True}

création d'une fonction afin de faire un test complet sur un exemple de l'échantillon de validation. La fonction suivante `run_single_experiment` réalise tout le pipeline suivant : 

- appel Gemini
- récupération de la réponse
- extraction des informations
- comparaison avec la référence
- création d'une ligne de résultat


In [81]:
def run_single_experiment(
    experiment_row,
    model_name="gemini-3.6-flash",
):
    call_result = call_gemini(
        prompt=experiment_row["prompt_text"],
        model_name=model_name,
    )

    parsed_result = parse_model_response(
        response=call_result["raw_response"],
        prompt_version=experiment_row[
            "prompt_version"
        ],
    )

    predicted_letter = parsed_result[
        "predicted_letter"
    ]

    # Une erreur technique ne doit pas être
    # comptée comme une mauvaise réponse
    if call_result["generation_error"] is not None:
        is_correct = None
    else:
        is_correct = (
            predicted_letter
            == experiment_row["reference_letter"]
        )

    return {
        "sample_id": experiment_row["sample_id"],
        "model_name": "Gemini",
        "model_version": model_name,
        "prompt_version": experiment_row[
            "prompt_version"
        ],
        "prompt_text": experiment_row[
            "prompt_text"
        ],
        "temperature": None,
        "raw_response": call_result[
            "raw_response"
        ],
        "predicted_letter": predicted_letter,
        "generated_justification": (
            parsed_result[
                "generated_justification"
            ]
        ),
        "declared_confidence": (
            parsed_result[
                "declared_confidence"
            ]
        ),
        "response_format_valid": (
            parsed_result[
                "response_format_valid"
            ]
        ),
        "reference_letter": experiment_row[
            "reference_letter"
        ],
        "is_correct": is_correct,
        "latency_seconds": call_result[
            "latency_seconds"
        ],
        "generation_error": call_result[
            "generation_error"
        ],
    }

In [82]:
# test sur une ligne 
first_experiment = df_experiments.iloc[0]

first_result = run_single_experiment(
    experiment_row=first_experiment,
)

first_result

{'sample_id': 'mcqu_validation_18257',
 'model_name': 'Gemini',
 'model_version': 'gemini-3.6-flash',
 'prompt_version': 'prompt_v1',
 'prompt_text': "Vous devez répondre à une question médicale à choix unique.\n\nCas clinique :\nUne patiente de 26 ans, enceinte de 20 semaines, vient en consultation pour sa visite du 5 ème mois. Le sérodiagnostic de toxoplasmose pratiqué 15 jours auparavant objective un taux d'lgG à 450 U.I., présence d'lgM. Le précédent sérodiagnostic pratiqué en début de grossesse à 6 semaines d'aménorrhée était négatif. Il s'agit d'une séroconversion. Un traitement à la Spiramycine (3 g/j) est instauré\n\nQuestion :\nLes risques de toxoplasmose congénitale pour le foetus sont de l'ordre de :\n\nPropositions :\nA. 90%\nB. 70%\nC. 50%\nD. 20%\nE. 0,5%\n\nSélectionnez une seule proposition parmi A, B, C, D ou E.\n\nRépondez uniquement avec la lettre correspondante.",
 'temperature': None,
 'raw_response': 'D',
 'predicted_letter': 'D',
 'generated_justification': None,

Testons les 3 prompts sur la même question

In [83]:
TEST_SAMPLE_ID = "mcqu_validation_18257"

df_same_question = (
    df_experiments[
        df_experiments["sample_id"]
        == TEST_SAMPLE_ID
    ]
    .sort_values("prompt_version")
    .reset_index(drop=True)
)

display(
    df_same_question[
        [
            "sample_id",
            "prompt_version",
            "reference_letter",
        ]
    ]
)

,sample_id,prompt_version,reference_letter
0,mcqu_validation_18257,prompt_v1,D
1,mcqu_validation_18257,prompt_v2,D
2,mcqu_validation_18257,prompt_v3,D


In [ ]:
prompt_test_results = []

for _, experiment_row in (
    df_same_question.iterrows()
):
    print(
        "Exécution de",
        experiment_row["prompt_version"],
        "..."
    )

    result = run_single_experiment(
        experiment_row=experiment_row,
        model_name="gemini-3.6-flash",
    )

    prompt_test_results.append(result)

    print(
        "Réponse :",
        result["predicted_letter"],
        "| correcte :",
        result["is_correct"],
        "| format valide :",
        result["response_format_valid"],
        "| latence :",
        round(
            result["latency_seconds"],
            2,
        ),
        "secondes",
    )

Exécution de prompt_v1 ...
Réponse : D | correcte : True | format valide : True | latence : 4.37 secondes
Exécution de prompt_v2 ...
Réponse : D | correcte : True | format valide : True | latence : 7.38 secondes
Exécution de prompt_v3 ...
Réponse : D | correcte : True | format valide : True | latence : 6.06 secondes


In [85]:
df_prompt_test_results = pd.DataFrame(
    prompt_test_results
)

display(
    df_prompt_test_results[
        [
            "sample_id",
            "prompt_version",
            "raw_response",
            "predicted_letter",
            "declared_confidence",
            "response_format_valid",
            "reference_letter",
            "is_correct",
            "latency_seconds",
            "generation_error",
        ]
    ]
)

,sample_id,prompt_version,raw_response,predicted_letter,declared_confidence,response_format_valid,reference_letter,is_correct,latency_seconds,generation_error
0,mcqu_validation_18257,prompt_v1,D,D,NaN,True,D,True,4.373327,None
1,mcqu_validation_18257,prompt_v2,Réponse : D\n\nJustification : Le risque de tr...,D,NaN,True,D,True,7.376003,None
2,mcqu_validation_18257,prompt_v3,Réponse : D\nJustification : Le risque de tran...,D,95.0,True,D,True,6.060755,None


In [86]:
display(df_prompt_test_results.loc[2,"raw_response"])

"Réponse : D\nJustification : Le risque de transmission materno-fœtale du toxoplasme augmente progressivement au cours de la grossesse avec l'âge gestationnel. Il est faible au premier trimestre (environ 10 à 15 %), moyen au deuxième trimestre (de l'ordre de 20 à 30 %, soit environ 20 % aux alentours de 20 semaines d'aménorrhée) et élevé au troisième trimestre (60 à 70 %).\nConfiance : 95"

Sélectionnons cinq questions différentes de manière reproductible et leur appliquer les 3 prompts différents afin de générer 15 questions. Celà nous permet d'avoir une cohérence sur le nombre de prompts utilisé d'éviter un nombre inégal d’expériences par prompt.

In [89]:
PILOT_SAMPLE_SIZE = 5
PILOT_RANDOM_SEED = 42

pilot_sample_ids = (
    df_prompt_sample[
        "sample_id"
    ]
    .sample(
        n=PILOT_SAMPLE_SIZE,
        random_state=PILOT_RANDOM_SEED,
    )
    .tolist()
)

df_pilot_experiments = (
    df_experiments[
        df_experiments["sample_id"].isin(
            pilot_sample_ids
        )
    ]
    .sort_values(
        [
            "sample_id",
            "prompt_version",
        ]
    )
    .reset_index(drop=True)
)

Générer les 15 appels de test en boucle

In [ ]:
pilot_results = []

total_experiments = len(
    df_pilot_experiments
)

for experiment_number, (_, row) in enumerate(
    df_pilot_experiments.iterrows(),
    start=1,
):
    print(
        f"[{experiment_number}/"
        f"{total_experiments}] "
        f"{row['sample_id']} - "
        f"{row['prompt_version']}"
    )

    result = run_single_experiment(
        experiment_row=row,
        model_name="gemini-3.6-flash",
    )

    pilot_results.append(result)

    print(
        "  Réponse :",
        result["predicted_letter"],
        "| correcte :",
        result["is_correct"],
        "| format :",
        result["response_format_valid"],
        "| latence :",
        round(
            result["latency_seconds"],
            2,
        ),
        "s",
    )

[1/15] mcqu_validation_1199 - prompt_v1
  Réponse : D | correcte : True | format : True | latence : 2.92 s None
[2/15] mcqu_validation_1199 - prompt_v2
  Réponse : D | correcte : True | format : True | latence : 4.57 s None
[3/15] mcqu_validation_1199 - prompt_v3
  Réponse : D | correcte : True | format : True | latence : 3.49 s None
[4/15] mcqu_validation_12087 - prompt_v1
  Réponse : D | correcte : True | format : True | latence : 5.63 s None
[5/15] mcqu_validation_12087 - prompt_v2
  Réponse : D | correcte : True | format : True | latence : 5.43 s None
[6/15] mcqu_validation_12087 - prompt_v3
  Réponse : D | correcte : True | format : True | latence : 4.2 s None
[7/15] mcqu_validation_21870 - prompt_v1
  Réponse : C | correcte : True | format : True | latence : 4.83 s None
[8/15] mcqu_validation_21870 - prompt_v2
  Réponse : C | correcte : True | format : True | latence : 4.34 s None
[9/15] mcqu_validation_21870 - prompt_v3
  Réponse : C | correcte : True | format : True | latence :

In [99]:
df_pilot_results = pd.DataFrame(
    pilot_results
)

df_pilot_results = df_pilot_results[
    result_columns
]

print(
    "Dimensions des résultats :",
    df_pilot_results.shape
)

display(
    df_pilot_results[
        [
            "sample_id",
            "prompt_version",
            "predicted_letter",
            "reference_letter",
            "is_correct",
            "response_format_valid",
            "declared_confidence",
            "latency_seconds",
            "generation_error",
        ]
    ]
)

Dimensions des résultats : (15, 15)


,sample_id,prompt_version,predicted_letter,reference_letter,is_correct,response_format_valid,declared_confidence,latency_seconds,generation_error
0,mcqu_validation_1199,prompt_v1,D,D,True,True,NaN,2.920376,None
1,mcqu_validation_1199,prompt_v2,D,D,True,True,NaN,4.570492,None
2,mcqu_validation_1199,prompt_v3,D,D,True,True,100.0,3.490313,None
3,mcqu_validation_12087,prompt_v1,D,D,True,True,NaN,5.631593,None
4,mcqu_validation_12087,prompt_v2,D,D,True,True,NaN,5.425145,None
5,mcqu_validation_12087,prompt_v3,D,D,True,True,100.0,4.198366,None
6,mcqu_validation_21870,prompt_v1,C,C,True,True,NaN,4.833937,None
7,mcqu_validation_21870,prompt_v2,C,C,True,True,NaN,4.341155,None
8,mcqu_validation_21870,prompt_v3,C,C,True,True,98.0,4.395822,None
9,mcqu_validation_25168,prompt_v1,A,B,False,True,NaN,8.289646,None


Enregistrant le test pour analyse

In [100]:
PILOT_RESULTS_PATH = (
    PROMPT_RESULTS_DIR
    / "gemini_prompt_pilot.parquet"
)

df_pilot_results.to_parquet(
    PILOT_RESULTS_PATH,
    index=False,
)

print(
    "Résultats enregistrés dans :",
    PILOT_RESULTS_PATH
)

Résultats enregistrés dans : c:\Users\MANEL\Dropbox\projet_evaluation_LLM_medicale\data\results\prompt_tests\gemini_prompt_pilot.parquet


Vérification rapide

In [101]:
pilot_summary = (
    df_pilot_results
    .groupby("prompt_version")
    .agg(
        number_of_experiments=(
            "sample_id",
            "count",
        ),
        accuracy=(
            "is_correct",
            "mean",
        ),
        valid_format_rate=(
            "response_format_valid",
            "mean",
        ),
        mean_latency_seconds=(
            "latency_seconds",
            "mean",
        ),
        technical_errors=(
            "generation_error",
            lambda series: (
                series.notna().sum()
            ),
        ),
    )
    .reset_index()
)

display(pilot_summary)

,prompt_version,number_of_experiments,accuracy,valid_format_rate,mean_latency_seconds,technical_errors
0,prompt_v1,5,0.8,1.0,4.894315,0
1,prompt_v2,5,0.8,1.0,6.598309,0
2,prompt_v3,5,0.8,1.0,6.342103,0


Le cas mcqu_validation_25168 est particulièrement intéressant :

Réponse attendue : B
Réponse produite : A
Confiance déclarée avec V3 : 95 %

Le modèle est donc fortement confiant dans une réponse incorrecte. C’est exactement le type de situation pertinent pour une étude sur les hallucinations et la mauvaise calibration de la confiance. Etudions ce cas.

In [102]:
sample_id = "mcqu_validation_25168"

question_context = (
    df_prompt_sample.loc[
        df_prompt_sample["sample_id"] == sample_id,
        "question_context",
    ]
    .iloc[0]
)

print("QUESTION ET CONTEXTE\n")
print(question_context)

QUESTION ET CONTEXTE

Question :
(cochez la réponse juste) Au niveau de la jonction neuromusculaire du muscle strié squelettique :

Propositions :
A. Les muscles striés squelettiques sont innervés par des motoneurones myélinisés, de grand calibre, dont le corps cellulaire est situé dans la corne antérieure de la moelle épinière et dans le tronc cérébral
B. Le nombre de fibres musculaires striées squelettiques innervées par un motoneuro- ne est variable
C. Le neurotransmetteur est l'adrénaline
D. Le potentiel d'action (PA) se propage par activation des canaux sodium dont l'ouverture dépend de la libération d'un médiateur intracellulaire libéré par le canal sodique précédent
E. Le curare, comme la toxine botulinique, se fixe aux récepteurs nicotiniques, empêchant ainsi leur activation


In [106]:
response = (
    df_pilot_results.loc[
        df_pilot_results["sample_id"] == sample_id,
        [
            "prompt_version",
            "predicted_letter",
            "reference_letter",
            "generated_justification",
            "declared_confidence",
        ],
    ]
)
display(response)
# justification :
print("\nJustification du LLM")
display(response.iloc[1, response.columns.get_loc("generated_justification")])

,prompt_version,predicted_letter,reference_letter,generated_justification,declared_confidence
9,prompt_v1,A,B,None,NaN
10,prompt_v2,A,B,Les muscles striés squelettiques sont innervés...,NaN
11,prompt_v3,A,B,Les muscles striés squelettiques sont innervés...,95.0



Justification du LLM


"Les muscles striés squelettiques sont innervés par les motoneurones alpha (de grand calibre et myélinisés), dont les corps cellulaires se situent dans la corne antérieure de la moelle épinière (pour l'innervation spinale) et dans le tronc cérébral (pour les nerfs crâniens). \n\nPour les autres propositions :\n- C est fausse : le neurotransmetteur de la jonction neuromusculaire est l'acétylcholine (et non l'adrénaline).\n- D est fausse : la propagation du potentiel d'action repose sur la dépolarisation membranaire ouvrant les canaux sodium voltage-dépendants, et non sur un médiateur intracellulaire.\n- E est fausse : la toxine botulinique inhibe la libération pré-synaptique d'acétylcholine (par clivage des protéines SNARE), contrairement au curare qui bloque de manière compétitive les récepteurs nicotiniques post-synaptiques."

Cette question montre justement pourquoi il ne faut pas assimiler automatiquement une réponse incorrecte à une hallucination.

La référence B est clairement défendable : le nombre de fibres musculaires innervées par un motoneurone varie selon le muscle. Les petites unités motrices contrôlent quelques fibres, tandis que les grandes peuvent en contrôler des centaines ou des milliers. OpenStax

Mais la proposition A est également très plausible :

- les axones moteurs périphériques sont myélinisés ;
- les motoneurones spinaux proviennent de la corne antérieure ;
- les motoneurones des nerfs crâniens moteurs proviennent du tronc cérébral. NCBI Bookshelf

#### Pourquoi MediQAl retient probablement B

La proposition B est incontestablement correcte :

- Le nombre de fibres musculaires innervées par un motoneurone est variable.

La proposition A contient des imprécisions :

- ce sont rigoureusement les axones des motoneurones qui sont myélinisés, pas les corps cellulaires ;
- tous les motoneurones ne sont pas nécessairement « de grand calibre » ;
- la formulation généralise surtout les propriétés des motoneurones alpha.

B est donc probablement la réponse attendue parce qu’elle est plus rigoureuse et sans ambiguïté.

#### Problème dans le raisonnement du LLM

La justification du modèle explique correctement pourquoi A est plausible et pourquoi C, D et E sont fausses. En revanche, elle ne discute absolument pas la proposition B.

| Dimension                            | Conclusion                |
| ------------------------------------ | ------------------------- |
| Réponse selon le benchmark           | Incorrecte                |
| Faits présents dans la justification | Globalement corrects      |
| Erreur de raisonnement               | Oui                       |
| Omission de la proposition B         | Oui                       |
| Confiance mal calibrée               | Oui, 95 % malgré l’erreur |
| Hallucination factuelle nette        | Non ou incertaine         |
| Question potentiellement ambiguë     | Oui                       |

Ce cas sera très intéressant dans le rapport : il montre qu’une évaluation automatique fondée uniquement sur la lettre de référence peut pénaliser une réponse médicalement plausible et surestimer le nombre d’hallucinations